<a href="https://colab.research.google.com/github/lxndrbnsv/cir-notebooks/blob/main/compile_rtn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Инструкция по использованию

## Подготовка файлов

Загрузите два Excel-файла в Google Colab. Для этого нажмите на иконку папки в левой панели, затем кнопку загрузки файлов:
- `moscow.xlsx` — данные по Москве
- `mo.xlsx` — данные по Московской области

Убедитесь, что названия файлов совпадают с константами `MOSCOW_RTN` и `MO_RTN` в ячейке **Константы**. При необходимости измените их.

## Запуск

Запустите все ячейки сразу через меню **Среда выполнения → Выполнить все** (или `Ctrl+F9`). Ячейки выполнятся последовательно в правильном порядке автоматически.

Либо запускайте ячейки вручную сверху вниз:
1. **Зависимости** — установка библиотек и импорты
2. **Константы** — пути к файлам и имя выходного файла
3. **Логика** — функции парсинга и объединения данных

## Требования к файлам

**moscow.xlsx** должен содержать ровно три столбца в порядке: организация, ФИО, должность — без лишних строк сверху.

**mo.xlsx** может содержать произвольное количество строк-заголовков сверху — код найдёт нужную строку автоматически по слову «Наименование».

## Результат

После выполнения всех ячеек в папке Colab появится файл `rtn.xlsx` с объединёнными данными из обоих источников. Скачать его можно через правый клик в панели файлов → *Скачать*.

Итоговый файл содержит четыре столбца:
- `company` — название организации
- `name` — ФИО сотрудника
- `position` — должность
- `source` — источник (`Москва` или `МО`)

# Зависимости

In [1]:
%%capture
!pip install openpyxl
!pip install pandas

In [2]:
import re

import pandas as pd

# Константы

**Файлы с данными РТН**

In [3]:
MOSCOW_RTN = "moscow.xlsx"  # Москва
MO_RTN = "mo.xlsx"  # Московская область

**Файл для сохранения результата слияния**

In [4]:
OUTPUT_FILE = "rtn.xlsx" # Файл с результатом

# Логика

Здесь мы парсим оба файла excel, чтобы получить файл в едином формате. Добавляем столбец "source" для указания источника: "Москва" или "МО".

**Парсинг файла с данными по Москве**

In [5]:
def parse_moscow(filepath: str) -> pd.DataFrame:
    df = pd.read_excel(filepath)
    df.columns = ["company", "name", "position"]
    df["source"] = "Москва"
    return df

In [6]:
moscow_df = parse_moscow(MOSCOW_RTN)

**Парсинг файла с данными по МО**

In [7]:
def parse_name_string(name_string: str) -> tuple:
    if not isinstance(name_string, str):
        return None, None, None
    parts = name_string.split("\n")
    name = parts[0].strip()
    if len(parts) < 2:
        return name, None, None
    position_exp = parts[1].strip()
    match = re.search(
        r"\s*[\d,./]+\s*(лет|год|года|мес\.?|месяц|месяца|месяцев).*$",
        position_exp,
        re.IGNORECASE,
    )
    exp = match.group(0).strip() if match else None
    position = position_exp[: match.start()].strip() if match else position_exp
    position = re.sub(r"[\s\d,./]+$", "", position).strip()
    return name, position, exp

In [8]:
def parse_mo(filepath: str) -> pd.DataFrame:
    df = pd.read_excel(filepath)
    header_row = df[
        df.apply(
            lambda row: row.astype(str).str.contains("Наименование").any(),
            axis=1
        )
    ].index[0]
    df = pd.read_excel(filepath, header=header_row + 1)
    df = df.drop(columns=df.columns[:2])  # Убираем лишние столбцы слева.
    df = df.iloc[:, :2]  # Убираем лишние столбцы справа.
    df.columns = ["company", "name_position_exp"]

    # Парсим столбец с именами и должностями.
    df[["name", "position"]] = df["name_position_exp"].apply(
        lambda x: pd.Series(parse_name_string(x)[:2])
    )
    df = df.drop(columns=["name_position_exp"])
    df["source"] = "МО"

    return df

In [9]:
%%capture

mo_df = parse_mo(MO_RTN)

**Объединяем два списка и формируем Excel-файл**

In [10]:
df = pd.concat([moscow_df, mo_df], ignore_index=True)

In [11]:
df.to_excel(OUTPUT_FILE)